# §29 — parallel_cubic: cubic'i sıralı z-taraması olmadan yeniden formüle etmek

**Otorite ön-kayıt `RESULTS.md` §29b'dedir.** Bu markdown kolaylık kopyasıdır.

**Fikir.** λ = 1/√(1+2η·z²) sıralı z-taraması gerektiriyor. Blok-donmuş form:
λ, her bloğun başındaki **gerçek** z'den hesaplanıp blok içinde sabit tutulur →
blok-içi hesap exp/GLA ile aynı paralel makineye çöker, yalnız bloklar arası
taşıma sıralı kalır. **Yeni hiperparametre yok.**

**Kazanç:** sıralı özel op yok → mobil ihraç engeli kalkar (raf koşulu #1 ve #2).

**Risk (dürüst):** §28a'daki kazanç *uyarlanabilirlikten* mi geliyordu, yoksa tam
kapalı-döngü dinamiğinden mi? Bilmiyoruz — bu deney onu ölçüyor.

**Hücre 2 bir DOĞRULAMA KAPISI** ve zaten geçti (RESULTS §29): `rec_block=1`'de
sıralı cubic ile bit-bit aynı (0.000e+00). Geçmezse deney koşulmaz.

---

## v3 (2026-08-01) — iki düzeltme, koşudan önce

**1. Kayıp artık stdout'tan kazınmıyor.** Eski hücre 3 `RE_VER` regex'iyle
`carry_curriculum.py`'nin ekran çıktısını okuyordu. O print satırı §35 v4'te
değişti → regex eşleşmez, deney sessizce "IRAKSADI" basardı. Artık script'in
yazdığı `{TAG}_valloss.csv` okunuyor.

**2. Birincil metrik `val_cross`.** §35a'da bulundu: eski karışık kayıp,
B chunk'ının **chunk-içi** çiftlerini de içeriyordu (P=6'da denetlenen 7
tokenın 6'sı chunk-içi). §28a'nın +0.484 nat'ı o karışık metrikte ölçüldü.
Bu koşuda ikisi de raporlanıyor: `val_cross` (temiz, **birincil**) ve
`val_loss` (karışık, §28a ile kıyaslanabilirlik için).

**Bunun bir yan sonucu var ve önceden söylenmeli:** bu koşu §28a'yı temiz
metrikte yeniden ölçüyor. §28a'nın etkisi **zayıflayabilir**. Öyle çıkarsa
öyle yazılır.

**3. Sıra seed-major.** Her seed için üç kol arka arkaya koşuluyor, böylece
yarıda kesilse bile elde **tam eşleşmiş çiftler** kalır. Eski sürüm kol-major
koşuyordu; yarıda kesilince eşleşme kaybolurdu.

**4. Cache silme kaldırıldı.** Eski hücre 3 her koşuda `parallel_cubic`
cache'ini siliyordu (bir kerelik tasarım değişikliği içindi) — oturumlar arası
devam etmeyi imkansız kılıyordu. Yerine sürüm damgalı cache.


In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys, re, json, math
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
ROOT = os.path.join(BASE,'eta_sweep'); os.makedirs(ROOT, exist_ok=True)   # §28 ile ayni -> exp/cubic cache'i kullanilir
print('repo:', REPO, '| cikti:', ROOT)

In [ ]:
# --- 2. DOGRULAMA KAPISI (deneyden ONCE implementasyonu sina) ---
# [§29 v2] Tasarim degisti: acik-dongu vekil -> BLOK-DONMUS GERCEK z.
# Bu sayede cok daha guclu bir dogrulama mumkun: rec_block=1'de parallel_cubic,
# sirali cubic_flux_chunked ile BIREBIR AYNI olmali (yaklasim hatasi yok).
import torch, time
from hfp.models.configuration_hfp import HFPConfig
from hfp.models.modeling_hfp import HFPForCausalLM

def build(mode, seed=0, rec_block=32):
    torch.manual_seed(seed)
    cfg = HFPConfig(vocab_size=164, hidden_size=64, num_hidden_layers=2,
                    num_attention_heads=2, intermediate_size=256, bulk_dim=32,
                    short_len=8, max_position_embeddings=264, local_window=8,
                    decay_mode=mode, rec_block=rec_block, write_rule="additive",
                    key_feature_map="dpfp", pe_period=256)
    return HFPForCausalLM(cfg).eval()

x = torch.randint(1, 100, (2, 256))

# T1 (EN KRITIK): rec_block=1 -> parallel_cubic == cubic_flux_chunked (ozdes)
mc = build("cubic_flux_chunked", 0, rec_block=1)
mp = build("parallel_cubic",     0, rec_block=1); mp.load_state_dict(mc.state_dict())
with torch.no_grad(): a = mc(x).logits; b = mp(x).logits
d1 = (a-b).abs().max().item()
print(f'T1 (rec_block=1 -> sirali cubic ile OZDES): max|fark| = {d1:.3e}  -> {"GECTI" if d1 < 1e-4 else "KALDI"}')

# T2: rec_block=32 -> yakin ama ayni degil (blok-donmus yaklasim devrede)
mc32 = build("cubic_flux_chunked", 0, rec_block=32)
mp32 = build("parallel_cubic",     0, rec_block=32); mp32.load_state_dict(mc32.state_dict())
with torch.no_grad(): a32 = mc32(x).logits; b32 = mp32(x).logits
d2 = (a32-b32).abs().max().item()
rel = d2 / (a32.abs().mean().item() + 1e-9)
print(f'T2 (rec_block=32 -> yaklasim): max|fark| = {d2:.3e} (goreli ~{rel:.2%}) -> bilgi amacli')

# T3: NaN yok, gradyan akiyor
mp3 = build("parallel_cubic", 1); mp3.train()
out = mp3(x, labels=x); out.loss.backward()
ge = [p.grad for n,p in mp3.named_parameters() if 'log_eta' in n and p.grad is not None]
t3 = bool(torch.isfinite(out.loss)) and all(torch.isfinite(g).all() for g in ge) and any(g.abs().sum()>0 for g in ge)
print(f'T3 (loss finite + log_eta gradyani akiyor): {"GECTI" if t3 else "KALDI"}  loss={out.loss.item():.3f}')

# T4: uzun dizide NaN dayanikliligi (eski tasarimi cokerten senaryo)
mp4 = build("parallel_cubic", 2); mp4.train()
xl_ = torch.randint(1,100,(2,1024))
o4 = mp4(xl_, labels=xl_); o4.loss.backward()
t4 = bool(torch.isfinite(o4.loss)) and all(torch.isfinite(p.grad).all()
        for p in mp4.parameters() if p.grad is not None)
print(f'T4 (L=1024 uzun dizi, NaN yok): {"GECTI" if t4 else "KALDI"}  loss={o4.loss.item():.3f}')

# T5: HIZ
xl = torch.randint(1,100,(4,512)); sp={}
for mode in ("cubic_flux_chunked","parallel_cubic","exp"):
    mm = build(mode,0)
    with torch.no_grad():
        mm(xl); t0=time.time()
        for _ in range(3): mm(xl)
        sp[mode]=(time.time()-t0)/3
    print(f'   {mode:>20}: {sp[mode]*1000:7.1f} ms/forward (B=4,L=512)')
print(f'   -> parallel_cubic, sirali cubic\'e gore {sp["cubic_flux_chunked"]/sp["parallel_cubic"]:.2f}x hizli')

GATE = (d1 < 1e-4) and t3 and t4
print('\n' + ('KAPI GECILDI -> deney kosulabilir.' if GATE else
      'KAPI KALDI -> IMPLEMENTASYON HATALI. Deneyi KOSMA.'))
assert GATE, 'Dogrulama kapisi basarisiz'

In [ ]:
# --- 3. DENEY: n eslesmis seed x 3 kol (exp / cubic sirali / parallel_cubic) ---
import csv, time
CACHE_VER = 'v3-valcross'      # cache damgasi; degisirse eski sonuclar yeniden kosulur
N_SEEDS  = 16
SEED_LO, SEED_HI = 0, 16       # oturuma sigmazsa bolerek kos: (0,6) -> (6,12) -> (12,16)
BASE_ENV = {**os.environ, 'PYTHONPATH': REPO,
            'CC_CARRY_MAX':'16','CC_STEPS':'1200','CC_CTX':'256','CC_P':'6',
            'CC_DIST_EVERY':'64','CC_BS':'8','CC_GAPS':'256','CC_TRIALS':'60',
            'CC_VAL_N':'64','CC_VAL_K':'2'}
ARMS = [('exp','exp_reference'), ('cubic_flux_chunked','cubic_eta_default'),
        ('parallel_cubic','parallel_cubic')]
for _m, _t in ARMS: os.makedirs(os.path.join(ROOT,_t), exist_ok=True)

def read_valloss(ck, mode, s):
    """Script'in yazdigi CSV'yi oku. stdout KAZIMA YOK — print formati degisebilir."""
    f = os.path.join(ck, f'carryv1_{mode}_s{s}_valloss.csv')
    if not os.path.exists(f): return None
    r = next(csv.DictReader(open(f)))
    return {'cross': float(r['val_cross']), 'inchunk': float(r['val_inchunk']),
            'mixed': float(r['val_loss']), 'nan': False, 'ver': CACHE_VER}

data = {t: {} for _m, t in ARMS}
_times = []
# SEED-MAJOR: yarida kesilse bile tam eslesmis ciftler kalir
for s in range(SEED_LO, SEED_HI):
    for mode, tag in ARMS:
        ck = os.path.join(ROOT, tag)
        cache = os.path.join(ck, f'result_s{s}.json')
        if os.path.exists(cache):
            d = json.load(open(cache))
            if d.get('ver') == CACHE_VER:
                data[tag][s] = d; continue
        env = {**BASE_ENV, 'HFP_CKPT_DIR': ck}
        env.pop('HFP_ETA_LOG_MIN', None); env.pop('HFP_ETA_LOG_MAX', None)
        print(f'[{tag} s{s}] ...', end=' ', flush=True)
        _t0 = time.time()
        r = subprocess.run([sys.executable,'review_scripts/carry_curriculum.py',mode,str(s),'6000'],
                           cwd=REPO, env=env, capture_output=True, text=True)
        _dt = time.time()-_t0; _times.append(_dt)
        d = read_valloss(ck, mode, s)
        if d is None:
            d = {'cross': None,'inchunk': None,'mixed': None,'nan': True,'ver': CACHE_VER}
            print('IRAKSADI/CSV YOK', flush=True)
            print('  --- son 15 satir stdout ---')
            print('\n'.join((r.stdout or '').strip().split('\n')[-15:]))
            print('  --- stderr ---'); print((r.stderr or '')[-800:])
        else:
            print(f"cross {d['cross']:.3f} | inchunk {d['inchunk']:.3f} | "
                  f"karisik {d['mixed']:.3f}  ({_dt/60:.1f} dk)", flush=True)
        json.dump(d, open(cache,'w')); data[tag][s] = d
        if len(_times) == 1:
            _kalan = (SEED_HI-SEED_LO)*len(ARMS) - 1
            print(f'\n  >>> SURE PROJEKSIYONU: ilk kol {_dt/60:.1f} dk -> kalan {_kalan} kol '
                  f'~{_kalan*_dt/3600:.1f} saat', flush=True)
            print('  >>> 12 saate sigmiyorsa DURDUR ve SEED_LO/SEED_HI ile bol.\n', flush=True)
json.dump(data, open(os.path.join(ROOT,'parallel_cubic_n16.json'),'w'), indent=2)
print(f'\nDENEY TAMAM — seed {SEED_LO}..{SEED_HI-1}')
if _times: print(f'toplam {sum(_times)/3600:.2f} saat, kol basina ort {sum(_times)/len(_times)/60:.1f} dk')

In [ ]:
# --- 4. ON-KAYITLI HUKUM (§29) ---
import statistics as st, math
from math import lgamma
def betacf(a,b,x):
    MAXIT,EPS,FPMIN=200,3e-12,1e-300
    qab,qap,qam=a+b,a+1,a-1; c=1.0; dd=1-qab*x/qap
    if abs(dd)<FPMIN: dd=FPMIN
    dd=1/dd; h=dd
    for mm in range(1,MAXIT+1):
        m2=2*mm
        aa=mm*(b-mm)*x/((qam+m2)*(a+m2)); dd=1+aa*dd; c=1+aa/c
        if abs(dd)<FPMIN: dd=FPMIN
        if abs(c)<FPMIN: c=FPMIN
        dd=1/dd; h*=dd*c
        aa=-(a+mm)*(qab+mm)*x/((a+m2)*(qap+m2)); dd=1+aa*dd; c=1+aa/c
        if abs(dd)<FPMIN: dd=FPMIN
        if abs(c)<FPMIN: c=FPMIN
        dd=1/dd; de=dd*c; h*=de
        if abs(de-1)<EPS: break
    return h
def tp(t,df):
    x=df/(df+t*t); a,b=df/2,0.5
    if x<(a+1)/(a+b+2):
        return math.exp(lgamma(a+b)-lgamma(a)-lgamma(b)+a*math.log(x)+b*math.log(1-x))*betacf(a,b,x)/a
    return 1-math.exp(lgamma(a+b)-lgamma(a)-lgamma(b)+b*math.log(1-x)+a*math.log(x))*betacf(b,a,1-x)/b

def paired(tag_a, tag_b, field='cross'):
    """(a - b): POZITIF = b daha iyi (kayip dusuk)."""
    A, B = data[tag_a], data[tag_b]
    ss = [s for s in sorted(set(A) & set(B)) if not A[s]['nan'] and not B[s]['nan']]
    d = [A[s][field] - B[s][field] for s in ss]
    n = len(d)
    if n < 3: return (float('nan'),)*3 + (n,) + (float('nan'),)*2
    t = st.mean(d) / (st.stdev(d)/math.sqrt(n))
    return st.mean(d), st.stdev(d), sum(1 for x in d if x > 0), n, t, tp(abs(t), n-1)

def table(field, baslik):
    print(f'\n=== {baslik} ===')
    for mode, tag in ARMS:
        L = [data[tag][s][field] for s in data[tag] if not data[tag][s]['nan']]
        nan = sum(1 for s in data[tag] if data[tag][s]['nan'])
        if L: print(f'  {tag:>20}: {st.mean(L):.4f}  (n={len(L)}, iraksama={nan})')

table('inchunk', 'TESHIS: chunk-ici kayip (arac ogrendi mi? sans ln(30)=3.4012)')
table('cross',   'BIRINCIL: cross-chunk kayip (yalniz hedef token)')
table('mixed',   'KARISIK metrik (§28a ile kiyaslanabilirlik icin)')

print('\n=== ON-KAYITLI HUKUM (§29b) ===')
for field, etiket in (('cross','BIRINCIL val_cross'), ('mixed','IKINCIL karisik metrik')):
    mp_, sp_, w_, n_, t_, p_ = paired('exp_reference', 'parallel_cubic', field)
    mc_, sc_, wc, nc, tc, pc = paired('exp_reference', 'cubic_eta_default', field)
    print(f'\n--- {etiket} ---')
    print(f'  cubic(sirali)  vs exp : {mc_:+.4f} nat, {wc}/{nc} seed, t={tc:+.2f}, p={pc:.4f}')
    print(f'  parallel_cubic vs exp : {mp_:+.4f} nat, {w_}/{n_} seed, t={t_:+.2f}, p={p_:.4f}')
    if field == 'cross':
        CROSS = (mp_, mc_, w_, n_, p_, wc, nc, pc)

mp_, mc_, w_, n_, p_, wc, nc, pc = CROSS
print('\n=== HUKUM (birincil metrik uzerinden) ===')
# ON-KAYIT: payda ICSEL, dis referans (§28a +0.484) yalnizca capraz kontrol.
if not (mc_ > 0 and pc < 0.05):
    print(f'  ARAC KAPISI GECILMEDI: bu kosuda cubic(sirali) exp\'i yenmedi')
    print(f'  (D={mc_:+.4f}, p={pc:.4f}). §28a temiz metrikte tekrarlanmadi ->')
    print('  parallel_cubic hakkinda HUKUM VERILEMEZ; korunacak bir fayda yok.')
    print('  Bu, §28a icin ONEMLI bir bulgudur ve oyle kaydedilir.')
else:
    keep = mp_/mc_*100
    print(f'  icsel referans: cubic(sirali)-exp = {mc_:+.4f} nat (p={pc:.4f})')
    print(f'  korunan fayda : {keep:.0f}%   |  parallel p={p_:.4f}')
    print(f'  (capraz kontrol: §28a karisik metrikte +0.484 nat olcmustu)')
    if keep >= 70 and p_ < 0.05:
        print('\n  => RAF KALKAR: paralel form faydanin >=%70\'ini koruyor ve anlamli.')
        print('     Raf kosulu #1 (paralel form) ve #2 (ozel op yok) kapanir.')
        print('     Kalan tek kosul #3: LM olceginde seyrek-rejim tekrari.')
    elif keep >= 30:
        print('\n  => KISMI: uyarlanabilirlik faydanin bir kismini tasiyor,')
        print('     kapali-dongu de katkili. Ikisi de belgelenir, raf karari ERTELENIR.')
    else:
        print('\n  => BASARISIZ: kazanc kapali-dongu geri beslemeye ozguymus.')
        print('     parallel_cubic reddedilir, cubic rafta kalir. Durustce boyle yazilir.')

